# 4장 1강: Attention 메커니즘 — 실습문제

## 실습 목표

- Attention과 Self-Attention의 관계를 설명합니다.
- Query·Key·Value의 역할과 배열 크기를 구분합니다.
- 내적 점수를 구하고 √d_k로 나눈 뒤 Softmax와 가중합을 적용합니다.
- Attention 가중치와 최종 출력이 각각 무엇을 뜻하는지 해석합니다.

## 실습 환경 / 데이터

- Python, pandas, NumPy를 사용합니다. 딥러닝 프레임워크는 필요하지 않습니다.
- 기존 **`jena_climate_2009_2016.csv`**를 노트북과 같은 폴더에 두고 그 폴더를 VS Code에서 열어 실행하세요. pandas와 NumPy가 설치된 Python 환경을 커널로 선택합니다.
- 원본은 420,551행, 15열이며, 이번에는 **처음 6행**만 읽습니다. 선택 구간은 10분 간격의 관측입니다.
- 입력 열은 기온 `T (degC)`와 상대습도 `rh (%)`입니다. 시간 `Date Time`은 어떤 관측값인지 표시하는 데 사용합니다.
- 실습은 첫 3개 관측, 과제는 다음 3개 관측을 사용합니다. 한 행은 한 관측 시점, 두 열은 그 시점의 두 특성입니다.
- 입력을 **[기온/10, 상대습도/100]**으로 만듭니다. 계산용 고정 배율 조정이며 원본 데이터는 바꾸지 않습니다. 평균·표준편차를 이용하는 표준화와는 다릅니다.
- 준비 셀부터 위에서 아래로 실행하세요. 계산에는 반올림 전 값을 사용하고 출력에서만 반올림합니다.

## 이번 강의의 범위

제공된 **4장 1강 목차: Attention과 Self-Attention, Query·Key·Value, Scaled Dot-Product Attention**을 기준으로 구성했습니다. 강의 본문은 별도로 제공되지 않았으므로 가중치와 작은 입력 구간은 개념 확인용 예시입니다.

이 자료는 세 관측값 사이의 Attention 계산을 확인합니다. **이미 관측한 구간 전체를 함께 읽는 예시**이며 미래 기온을 예측하는 모델을 학습하지 않습니다. 각 행은 같은 구간의 세 시점을 모두 참고할 수 있습니다. 시간 문자열은 Q·K·V에 들어가지 않으므로 이 계산만으로 관측 간격이나 선후관계까지 학습했다고 해석하지 않습니다.

Q·K·V를 만드는 가중치는 고정해서 제공합니다. 함수 작성, 모델 학습, 미분, Multi-Head Attention, Transformer 전체 구조, 위치 정보 추가, 마스킹의 구현은 요구하지 않습니다. 이번에는 Attention 한 번의 계산과 해석에 집중합니다.

## 실습 구성

| 구분 | 문제 | 할 일 |
|---|---|---|
| 실습 1 | 1-1 | 기상 데이터에서 Q·K·V 만들기 |
| 실습 2 | 2-1 | Scaled Dot-Product Attention 단계별 계산 |
| 실습 2 | 2-2 | 크기 조정 전후의 가중치 비교 |
| 기초 과제 | 3-1 | 다음 3개 관측값에 같은 계산 적용 |

**실습 2개 + 기초 과제 1개**이며, 마지막에는 **실습 마무리 복습 질문 6개**가 있습니다. 실습 2를 전체 계산과 크기 조정의 의미로 나눴습니다. 과제는 같은 식을 다른 구간에 적용하는 필수 수준입니다.

## 시작 전에 읽을 용어

| 용어 | 쉽게 풀어쓴 뜻 | 이번 예시 |
|---|---|---|
| Attention | Query와 Key를 비교해 반영 비중을 정하고 Value를 가중합하는 방식 | 다른 관측 정보를 어느 비중으로 섞을지 계산 |
| Self-Attention | 같은 입력 묶음에서 Q·K·V를 만드는 Attention | 같은 기상 관측 3행에서 모두 생성 |
| Query, Q | 정보를 모으는 기준이 되는 표현 | 현재 행이 각 Key와 비교될 때 쓰는 숫자 2개 |
| Key, K | Query와 비교되어 점수를 만드는 표현 | 참고할 각 관측 행의 숫자 2개 |
| Value, V | 계산된 비중으로 실제 합쳐질 정보 | 각 행의 기온/10 값 1개 |
| 내적 점수 | Q와 K의 같은 위치를 곱해 더한 값 | q의 두 값과 k의 두 값으로 계산 |
| d_k | Key 한 개를 표현하는 숫자 개수. Q도 같은 차원 | 2 |
| Scaling | 내적 점수를 √d_k로 나누는 것 | √2로 나눔 |
| Softmax | 점수를 양수의 비중으로 바꾸고 합을 1로 맞추는 함수 | Query 한 행마다 적용 |
| Attention 가중치 A | 각 Value를 얼마나 반영할지 나타내는 비중 | 한 행에 가중치 3개 |
| 출력 O | A로 V를 가중합해 얻은 결과 | Query마다 결과 숫자 1개 |

Attention은 상위 개념이고 Self-Attention은 그중 한 방식입니다. 일반적인 Attention에서는 Query의 입력 출처와 Key·Value의 입력 출처가 다를 수도 있습니다. Self-Attention의 ‘Self’는 **같은 입력에서 가져온다**는 뜻이며, 자기 행 하나만 보거나 Q=K=V여야 한다는 뜻이 아닙니다.


### 전체 계산식

$$Q=XW_Q,\quad K=XW_K,\quad V=XW_V$$
$$S=QK^T,\quad Z=S/\sqrt{d_k},\quad A=\operatorname{Softmax}(Z),\quad O=AV$$

Softmax는 각 행에서 계산합니다.

$$A_{ij}=\frac{\exp(Z_{ij})}{\sum_j\exp(Z_{ij})}$$

`i`는 정보를 모으는 Query 행, `j`는 참고할 Key·Value 행입니다. 분모는 **같은 i행의 모든 j**에 대해 더합니다. 각 Query가 참고하는 값들의 비중 합을 1로 만드는 것입니다.

표기 `@`는 행렬곱, `.T`는 행과 열을 바꾸는 전치, `np.sqrt()`는 제곱근입니다. 이번 내용은 앞서 배운 가중합과 행렬곱을 사용합니다.


### 왜 Attention을 배우나요?

여러 관측값을 하나로 요약할 때 모든 값을 똑같이 반영하는 평균을 사용할 수 있습니다. Attention은 **어떤 관측을 기준으로 정보를 모으는지에 따라, 참고할 값들의 비중을 계산하는 방식**입니다. 같은 세 관측을 참고하더라도 기준이 되는 Query가 달라지면 비중과 요약 결과가 달라질 수 있습니다.

예를 들어 세 값이 10, 20, 30이고 반영 비중이 0.2, 0.3, 0.5라면 `0.2×10 + 0.3×20 + 0.5×30 = 23`입니다. 이처럼 비중을 곱해 더하는 것을 **가중합**이라고 합니다. 여기서는 비중이 모두 0 이상이고 합이 1이므로 가중평균으로도 읽을 수 있습니다. Attention은 이 비중을 Q와 K의 비교로 계산합니다. 위의 0.2·0.3·0.5는 계산 방법을 설명하기 위한 숫자이며 실제 실습의 정답은 아닙니다.


### 벡터·행렬·기호부터 읽기

벡터는 숫자를 순서대로 모은 것이고, 행렬은 그런 숫자를 행과 열로 배열한 것입니다. 이번에는 한 시점의 `[기온/10, 상대습도/100]`이 숫자 두 개를 가진 벡터이며, 세 시점을 세 행으로 모으면 입력 행렬 X가 됩니다.

| 표현 | 쉽게 읽기 | 이번 값 또는 역할 |
|---|---|---|
| `(3, 2)` | 3행, 2열 | 관측 3개, 관측마다 특성 2개 |
| Q, K, V | 여러 관측의 표현을 모은 행렬 | 서로 다른 세 가지 계산 역할 |
| q1, k1, v1 | 각 행렬의 첫 번째 행 | 00:10 관측에서 만든 표현 |
| $W_Q,W_K,W_V$ | Q·K·V를 만드는 가중치 행렬 | 입력의 각 성분을 어떻게 결합할지 정함 |
| $K^T$ | K의 전치 | 행을 열로 바꾸어 모든 Q·K 쌍을 비교할 준비 |
| $d_k$ | Key 한 행의 성분 수 | 이 예시는 2이며, 관측 수 3과 다름 |
| $\sqrt{d_k}$ | d_k의 제곱근 | 제곱하면 d_k가 되는 양수; √2 ≈ 1.414214 |
| S, Z, A, O | 점수, 조정한 점수, 비중, 출력 | 서로 다른 계산 단계의 결과 |
| $A_{ij}$ | A의 i번째 행, j번째 열 | i번째 Query가 j번째 Value에 주는 비중 |
| $\sum_j$ | j를 바꾸며 모두 더하기 | 같은 Query 행의 세 값을 합함 |
| $\exp(z)$ | 지수함수 e의 z제곱 | Softmax에서 점수를 양수로 바꾸는 계산 |

`exp`의 e는 약 2.718인 수입니다. `exp(0)=1`이고 입력이 커질수록 결과도 커집니다. 이 수업에서는 함수값을 손으로 구하지 않고 `np.exp()`나 제공 Softmax 함수로 확인하면 됩니다.

수식에서는 첫 행을 q1이라고 부르지만, Python의 첫 행 번호는 0이라 `Q[0]`으로 선택합니다. `A[0, 2]`는 첫 번째 Query가 세 번째 Value에 주는 비중입니다. 아래첨자 i·j는 제곱을 뜻하지 않고 행·열 번호를 구분하는 표시입니다.


### 주제별 학습 위치

| 강의 주제 | 확인할 문제 | 설명할 수 있어야 하는 내용 |
|---|---|---|
| Attention과 Self-Attention | 실습 1-1 Q1, 실습 2-1 | 비중으로 정보 결합, 같은 입력에서 Q·K·V 생성 |
| Query·Key·Value | 실습 1-1, 과제 3-1 | 각 표현의 역할, 크기, 점수와 결합에 쓰이는 위치 |
| Scaled Dot-Product Attention | 실습 2-1·2-2, 과제 3-1 | 내적 → 크기 조정 → Softmax → Value 가중합 |

문제는 수식을 암기하여 새 알고리즘을 구현하기보다, 제공된 계산을 실행하고 각 단계의 의미를 설명하는 기초 수준입니다.


## 실습 준비 — 두 코드 셀을 그대로 실행하세요

첫 셀은 데이터와 가중치를 준비합니다. 한 입력 행을 `[a, b]=[기온/10, 상대습도/100]`이라고 하면 다음과 같이 변환됩니다.

| 변환 | 가중치 행렬 | 한 행의 결과 |
|---|---|---|
| Query | WQ = [[1, 0], [0, 1]] | [a, b] |
| Key | WK = [[1, 0], [0, 0.5]] | [a, 0.5b] |
| Value | WV = [[1], [0]] | [a] |

가중치는 계산을 따라가기 쉽게 정한 값이며 학습한 결과가 아닙니다. 일반적인 모델에서는 이런 변환 가중치를 학습합니다. **WQ·WK·WV와 Attention 가중치 A는 다릅니다.** 전자는 Q·K·V를 만드는 파라미터이고, 후자는 현재 입력의 점수에서 계산되는 반영 비중입니다.

두 번째 셀은 **행별 Softmax 제공 함수**입니다. 함수를 다시 작성하는 것은 문제가 아닙니다. `axis=1`은 각 행의 열 방향으로 계산한다는 뜻입니다. `keepdims=True`는 결과를 세로 한 열 모양으로 유지하여 원래 행렬의 각 행에 맞춰 계산하게 합니다.

함수는 지수 계산 전에 각 행의 최댓값을 뺍니다. 같은 행의 모든 점수에서 같은 수를 빼도 Softmax 비율은 같으며, 큰 수의 지수 계산을 피하는 데 도움이 됩니다. 증명은 요구하지 않습니다.

### 코드와 행렬 크기 읽기

- `=`는 오른쪽 계산 결과를 왼쪽 이름에 저장한다는 뜻입니다. `Q = X @ WQ`는 행렬곱 결과를 Q에 저장합니다.
- `@`는 행렬곱입니다. 왼쪽 행과 오른쪽 열에서 대응하는 숫자끼리 곱하고 더해 출력 한 칸을 만듭니다. 배열의 같은 위치끼리 곱하는 `*`와 다릅니다.
- `(3, 2) @ (2, 2)`에서 가운데 2가 같으므로 곱할 수 있고 결과는 바깥 크기 `(3, 2)`입니다. `(3, 2) @ (2, 1)`의 결과는 `(3, 1)`입니다.
- `K.shape[1]`은 K의 열 개수입니다. `shape[0]`은 행 개수이므로 여기서는 d_k를 구할 때 사용하지 않습니다.
- `.T`는 전치입니다. 예를 들어 행이 `[1, 2]`, `[3, 4]`인 행렬을 전치하면 행이 `[1, 3]`, `[2, 4]`가 됩니다.
- `axis=1`로 합을 구하면 각 행의 값을 가로 방향으로 더합니다. 이번에는 Query별 비중 합을 확인합니다.
- `np.round(값, 6)`은 소수 여섯 자리로 반올림한 표시용 값을 만듭니다. 중간 계산에는 반올림 전 숫자를 사용합니다.

준비 셀을 실행한 뒤 실습 1부터 차례로 실행하세요. 뒤 문제의 `Q`, `scores`, `weights` 등은 앞 문제에서 만든 결과를 사용합니다.


### 처음 코드를 읽는 학생을 위한 안내

| 표현 | 읽는 방법 | 이번 실습의 예 |
|---|---|---|
| `# 설명` | 사람이 읽는 주석이며 계산하지 않습니다. | 각 줄의 목적을 알려 줍니다. |
| `이름 = 계산` | 오른쪽 결과를 왼쪽 이름에 저장합니다. | `scores = Q @ K.T`는 점수 행렬을 저장합니다. |
| `np.array([...])` | 여러 숫자를 계산 가능한 배열로 만듭니다. | 가중치 행렬을 준비합니다. |
| `[[...], [...]]` | 안쪽 목록 하나가 한 행인 2차원 배열입니다. | `[[1], [0]]`은 2행·1열입니다. |
| `.shape` | 배열의 행·열 크기를 확인합니다. | `(3, 2)`는 3행·2열입니다. |
| `[0]`, `[0, 0]` | 첫 행 또는 첫 행·첫 열을 선택합니다. | Python의 위치 번호는 0부터 시작합니다. |
| `[:, 0]` | 모든 행에서 첫 열만 선택합니다. | V에 있는 세 숫자를 한 줄의 배열로 꺼냅니다. |
| `def 함수(...):` | 나중에 사용할 처리 방법을 정의합니다. | `softmax_rows`의 계산 순서를 준비합니다. |
| `return 결과` | 함수의 계산 결과를 호출한 곳에 돌려줍니다. | `weights = softmax_rows(...)`로 결과를 받습니다. |
| `print(...)` | 값을 화면에 표시합니다. | 표와 행렬의 결과를 확인합니다. |

준비 셀은 그대로 실행하고, 실습 1부터 순서대로 풀어 보세요. 실습 2는 앞에서 만든 Q·K·V를 사용합니다. `NameError`로 변수 이름을 찾지 못한다면, 필요한 값을 만드는 앞 셀을 실행했는지 먼저 확인하세요.

**배열의 값과 배열의 크기를 구분하세요.** `Q[0]`은 숫자들이 들어 있는 첫 행이고 `Q.shape`는 행·열 개수입니다. `K.shape[1]`의 1은 두 번째 항목인 **열 개수**를 고르는 위치 번호이며, Key의 첫 값이나 두 번째 관측값을 고르는 코드가 아닙니다.


In [1]:
# 표 형태의 데이터를 다루는 pandas를 pd라는 이름으로 불러옵니다.
import pandas as pd
# 숫자 배열 계산을 위해 NumPy를 np라는 이름으로 불러옵니다.
import numpy as np

# 노트북과 같은 폴더의 CSV에서 앞 여섯 행을 읽습니다.
df = pd.read_csv("data/jena_climate_2009_2016.csv", nrows=6)
# 관측 시각을 날짜·시간 자료로 바꿉니다. 일.월.년 순서입니다.
times = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
# 기온, 상대습도 순으로 두 열을 선택해 6행·2열 배열로 만듭니다.
raw = df[["T (degC)", "rh (%)"]].to_numpy(dtype=float)
# 각 행의 첫 열은 10으로, 둘째 열은 100으로 나눕니다.
X_all = raw / np.array([10.0, 100.0])
# 위치 0~2의 세 관측을 실습 입력으로 선택합니다.
X = X_all[:3]
# 위치 3~5의 세 관측을 과제 입력으로 선택합니다.
X_home = X_all[3:6]

# 두 입력 성분을 그대로 Q의 두 성분으로 만드는 예시 가중치입니다.
WQ = np.array([[1.0, 0.0], [0.0, 1.0]])
# 둘째 입력 성분을 절반으로 바꾸는 K용 가중치입니다.
WK = np.array([[1.0, 0.0], [0.0, 0.5]])
# 첫 입력 성분만 V에 남기는 2행·1열 가중치입니다.
WV = np.array([[1.0], [0.0]])

# 읽은 표의 행·열 개수를 확인합니다.
print("읽은 데이터 크기:", df.shape)
# 시간·기온·습도만 화면에 보여 줍니다.
print(df[["Date Time", "T (degC)", "rh (%)"]].to_string(index=False))
# 시각 차이를 초로 바꾸고 60으로 나누어 분 단위로 확인합니다.
print("선택 구간 간격(분):", times.diff().dropna().dt.total_seconds().unique() / 60)
# 첫 sum은 열별 빈칸 수, 둘째 sum은 두 열의 합계를 구합니다.
print("선택 두 열의 결측값 수:", df[["T (degC)", "rh (%)"]].isna().sum().sum())
# 실습 배열의 각 행이 한 시점인지 확인합니다.
print("실습 X:\n", X)
# 과제 배열은 준비 단계에서 이미 배율 조정이 끝났습니다.
print("과제 X_home:\n", X_home)
# 세 가중치 행렬의 크기를 확인합니다.
print("가중치 크기:", WQ.shape, WK.shape, WV.shape)


읽은 데이터 크기: (6, 15)
          Date Time  T (degC)  rh (%)
01.01.2009 00:10:00     -8.02    93.3
01.01.2009 00:20:00     -8.41    93.4
01.01.2009 00:30:00     -8.51    93.9
01.01.2009 00:40:00     -8.31    94.2
01.01.2009 00:50:00     -8.27    94.1
01.01.2009 01:00:00     -8.05    94.4
선택 구간 간격(분): [10.]
선택 두 열의 결측값 수: 0
실습 X:
 [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]]
과제 X_home:
 [[-0.831  0.942]
 [-0.827  0.941]
 [-0.805  0.944]]
가중치 크기: (2, 2) (2, 2) (2, 1)


In [2]:
# 점수 행렬의 각 행에 Softmax를 적용하는 제공 함수입니다.
def softmax_rows(scores):
    # 행마다 최댓값을 찾아 그 행의 모든 점수에서 뺍니다. 비중은 바뀌지 않습니다.
    shifted = scores - np.max(scores, axis=1, keepdims=True)
    # 각 숫자에 지수 함수를 적용합니다. exp(0)=1입니다.
    exp_scores = np.exp(shifted)
    # 각 행의 지수값 합으로 나누어 비중 합을 1로 맞춥니다.
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


print("행별 Softmax 함수 준비 완료")


행별 Softmax 함수 준비 완료


---

## 실습 1. Attention과 Query·Key·Value


### 문제 1-1. 같은 입력에서 Q·K·V 만들기

#### 문제 설명

입력 X는 세 관측 시점의 기온과 상대습도를 담고 있습니다. 준비된 세 가중치 행렬을 사용해 Q·K·V를 각각 만드세요.

#### 요구사항

1. `Q=X@WQ`, `K=X@WK`, `V=X@WV`를 계산하세요.
2. X·Q·K·V의 값과 크기를 출력하세요.
3. 첫 번째 관측 행으로 만든 q1·k1·v1을 직접 계산하세요. 대문자는 여러 행을 모은 행렬, 소문자는 그중 한 행을 뜻합니다.

#### 작성 도움말

`print(Q)`는 값, `print(Q.shape)`는 행·열 크기를 보여줍니다. 첫 행은 `Q[0]`입니다. 첫 입력은 `[-0.802, 0.933]`입니다.

#### 해석 질문

**Q1.** Attention과 Self-Attention은 어떤 관계인가요? 이번 계산이 Self-Attention인 이유는 무엇인가요?

**Q2.** Q·K·V의 크기와 첫 행의 값은 각각 무엇인가요? Q와 K는 두 값인데 V는 한 값이어도 되나요?

**Q3.** Q·K·V는 각각 어떤 역할을 하나요? 이번 가중치 행렬은 학습한 결과인가요? WQ·WK·WV와 이후에 구할 A는 어떻게 다른가요?

#### 제출 결과

행렬 값·크기, 첫 행 변환 계산, Q1~Q3 답변.


In [3]:
# 제공된 가중치로 Q, K, V를 만드세요.
Q = X @ WQ
K = X @ WK
V = X @ WV

# 각 값과 shape, 첫 행을 출력하세요.
print("X:", X, X.shape)
print("Q:", Q, Q.shape)
print("K:", K, K.shape)
print("V:", V, V.shape)
print("첫 행 q1, k1, v1:", Q[0], K[0], V[0])

X: [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]] (3, 2)
Q: [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]] (3, 2)
K: [[-0.802   0.4665]
 [-0.841   0.467 ]
 [-0.851   0.4695]] (3, 2)
V: [[-0.802]
 [-0.841]
 [-0.851]] (3, 1)
첫 행 q1, k1, v1: [-0.802  0.933] [-0.802   0.4665] [-0.802]


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 한 행 [a, b] = [-0.802, 0.933]에 가중치를 곱한다.
  - q1 = [-0.802×1 + 0.933×0, -0.802×0 + 0.933×1] = [-0.802, 0.933]
  - k1 = [-0.802×1 + 0.933×0, -0.802×0 + 0.933×0.5] = [-0.802, 0.4665]
  - v1 = [-0.802×1 + 0.933×0] = [-0.802]
  - 크기: X (3, 2), Q (3, 2), K (3, 2), V (3, 1)

**Q1. Attention과 Self-Attention은 어떤 관계인가요? 이번 계산이 Self-Attention인 이유는 무엇인가요?**

- 답변: Attention은 Query와 Key를 비교해 정한 비중으로 Value를 합치는 상위 개념이고, Self-Attention은 Q·K·V를 같은 입력 묶음에서 만드는 한 방식이다. 이번에는 Q·K·V를 모두 같은 X에서 만들었으므로 Self-Attention이다.

**Q2. Q·K·V의 크기와 첫 행의 값은 각각 무엇인가요? Q와 K는 두 값인데 V는 한 값이어도 되나요?**

- 답변: Q (3, 2), K (3, 2), V (3, 1)이고 첫 행은 q1 = [-0.802, 0.933], k1 = [-0.802, 0.4665], v1 = [-0.802]다. Q와 K는 내적을 위해 성분 수가 같아야 하지만 V는 비중으로 합쳐지는 값이라 성분이 한 개여도 된다.

**Q3. Q·K·V는 각각 어떤 역할을 하나요? 이번 가중치 행렬은 학습한 결과인가요? WQ·WK·WV와 이후에 구할 A는 어떻게 다른가요?**

- 답변: Q는 정보를 모으는 기준, K는 Q와 비교되어 점수를 만드는 표현, V는 비중에 따라 실제로 합쳐지는 정보다. 이번 가중치는 계산을 따라가기 쉽게 고정한 값이며 학습 결과가 아니다. WQ·WK·WV는 Q·K·V를 만드는 파라미터이고, A는 현재 입력의 점수에서 계산되는 반영 비중이다.


---

## 실습 2. Scaled Dot-Product Attention


### 문제 2-1. 내적부터 Value 가중합까지 계산하기

#### 문제 설명

실습 1의 Q·K·V를 이용해 Attention의 네 단계를 계산하세요.

$$S=QK^T\quad\rightarrow\quad Z=S/\sqrt{d_k}\quad\rightarrow\quad A=\operatorname{Softmax}(Z)\quad\rightarrow\quad O=AV$$

이번에는 d_k=2입니다. A의 i행 j열은 **i번째 Query가 j번째 Value에 주는 비중**입니다. K는 그 비중의 점수를 정하는 데 사용합니다.

#### 요구사항

1. `scores=Q@K.T`로 모든 Query·Key 쌍의 내적을 구하세요.
2. `d_k=K.shape[1]`로 Key의 성분 수를 확인하고 `scaled_scores=scores/np.sqrt(d_k)`를 계산하세요.
3. 제공 함수 `softmax_rows(scaled_scores)`로 `weights`를 구하세요.
4. `output=weights@V`로 Value를 가중합하세요.
5. 네 단계의 값과 크기, 가중치의 행별 합을 출력하세요.
6. 첫 Query 행의 점수 3개와 최종 출력 하나를 직접 대입해 계산하세요. Softmax 숫자는 제공 함수로 확인해도 됩니다.

#### 작성 도움말

`weights.sum(axis=1)`로 각 행의 가중치 합을 구합니다. 첫 Query의 출력은 `weights[0,0]*V[0,0]+weights[0,1]*V[1,0]+weights[0,2]*V[2,0]`입니다. 코드의 첫 행·첫 열 번호는 0입니다.

#### 해석 질문

**Q1.** 첫 Query와 세 Key의 내적 점수는 각각 얼마인가요? 첫 내적의 곱셈·덧셈도 쓰세요.

**Q2.** 첫 Query의 Attention 가중치 3개와 그 합은 얼마인가요? 세 행렬 S·Z·A의 크기와 최종 O의 크기는 무엇인가요?

**Q3.** 첫 Query의 가중합 결과는 얼마인가요? Attention 가중치는 양수인데 출력이 음수일 수 있는 이유는 무엇인가요?

**Q4.** 첫 Query가 가장 큰 비중을 주는 관측은 무엇인가요? 그 값이 가장 크다는 사실만으로 해당 시점이 미래 기온의 원인이라고 결론 내릴 수 있나요?

#### 제출 결과

단계별 행렬·크기, 행별 가중치 합, 첫 행 대입 계산, Q1~Q4 답변.


In [4]:
# d_k는 Key 한 행의 성분 수입니다.
d_k = K.shape[1]
scores = Q @ K.T
scaled_scores = scores / np.sqrt(d_k)
weights = softmax_rows(scaled_scores)
output = weights @ V

# 각 값과 shape, weights.sum(axis=1)을 출력하세요.
print("d_k:", d_k)
print("scores:", np.round(scores, 6), scores.shape)
print("scaled_scores:", np.round(scaled_scores, 6), scaled_scores.shape)
print("weights:", np.round(weights, 8), weights.shape)
print("행별 가중치 합:", weights.sum(axis=1))
print("output:", np.round(output, 6), output.shape)
# 첫 출력에 대한 Value별 기여도도 확인하세요.
print("첫 출력의 Value별 기여도:", np.round(weights[0] * V[:, 0], 6), "합:", round(np.sum(weights[0] * V[:, 0]), 6))

d_k: 2
scores: [[1.078448 1.110193 1.120546]
 [1.110193 1.143459 1.154204]
 [1.120546 1.154204 1.165062]] (3, 3)
scaled_scores: [[0.762578 0.785025 0.792345]
 [0.785025 0.808548 0.816145]
 [0.792345 0.816145 0.823823]] (3, 3)
weights: [[0.32755578 0.33499149 0.33745273]
 [0.32728819 0.33507812 0.33763369]
 [0.32721829 0.33509955 0.33768215]] (3, 3)
행별 가중치 합: [1. 1. 1.]
output: [[-0.8316  ]
 [-0.831612]
 [-0.831615]] (3, 1)
첫 출력의 Value별 기여도: [-0.2627   -0.281728 -0.287172] 합: -0.8316


#### 답변 작성란

**수식 및 계산 과정**

- 작성: d_k = 2
  - s11 = (-0.802)(-0.802) + (0.933)(0.4665) = 0.643204 + 0.435245 = 1.078448
  - s12 = (-0.802)(-0.841) + (0.933)(0.467) = 1.110193, s13 = (-0.802)(-0.851) + (0.933)(0.4695) = 1.120546
  - Z 첫 행 = S 첫 행 / √2 = [0.762578, 0.785025, 0.792345]
  - A 첫 행 = Softmax(Z 첫 행) = [0.32755578, 0.33499149, 0.33745273]
  - O 첫 값 = 0.32755578×(-0.802) + 0.33499149×(-0.841) + 0.33745273×(-0.851) = -0.262700 - 0.281728 - 0.287172 = -0.831600

**Q1. 첫 Query와 세 Key의 내적 점수는 각각 얼마인가요? 첫 내적의 곱셈·덧셈도 쓰세요.**

- 답변: 세 내적 점수는 1.078448, 1.110193, 1.120546이다. 첫 내적은 (-0.802)×(-0.802) + 0.933×0.4665 = 0.643204 + 0.435245로 계산한다.

**Q2. 첫 Query의 Attention 가중치 3개와 그 합은 얼마인가요? 세 행렬 S·Z·A의 크기와 최종 O의 크기는 무엇인가요?**

- 답변: 첫 Query의 가중치는 0.32755578, 0.33499149, 0.33745273이고 합은 1이다. S·Z·A의 크기는 모두 (3, 3)이고 O의 크기는 (3, 1)이다.

**Q3. 첫 Query의 가중합 결과는 얼마인가요? Attention 가중치는 양수인데 출력이 음수일 수 있는 이유는 무엇인가요?**

- 답변: 가중합 결과는 -0.831600이다. 가중치는 양수지만 합쳐지는 Value가 -0.802, -0.841, -0.851로 모두 음수이므로 그 가중평균인 출력도 음수가 된다.

**Q4. 첫 Query가 가장 큰 비중을 주는 관측은 무엇인가요? 그 값이 가장 크다는 사실만으로 해당 시점이 미래 기온의 원인이라고 결론 내릴 수 있나요?**

- 답변: 가장 큰 비중 0.33745273을 주는 관측은 세 번째인 00:30이다. 이 비중은 고정 가중치로 계산한 비교 결과일 뿐이고 세 비중의 차이도 작으므로 해당 시점이 미래 기온의 원인이라고 결론 내릴 수 없다. 미래 기온을 예측하거나 인과를 검증한 계산도 아니다.


### 문제 2-2. 왜 내적 점수를 √d_k로 나눌까요?

#### 문제 설명

문제 2-1의 `scores`를 그대로 사용해, 크기 조정 없이 Softmax를 적용한 경우와 비교하세요. 새로운 데이터나 가중치는 사용하지 않습니다.

- 크기 조정 있음: `softmax_rows(scores / sqrt(d_k))`
- 크기 조정 없음: `softmax_rows(scores)`

두 경우 모두 Softmax를 적용합니다. 이번에 비교할 것은 **내적 점수를 √d_k로 나누는 단계의 유무**입니다.

#### 요구사항

1. `weights_no_scale=softmax_rows(scores)`를 계산하세요.
2. 첫 Query의 조정 전후 가중치를 표로 비교하세요.
3. 두 경우의 첫 행 최댓값과 행 합을 출력하세요.
4. 크기 조정의 목적과 이번 숫자의 차이를 설명하세요.

#### 해석 질문

**Q1.** 이번에는 왜 √3이나 √1이 아니라 √2로 나누나요? 첫 행 가중치의 최댓값은 조정 전후에 각각 얼마인가요?

**Q2.** Scaled Dot-Product Attention에서 크기 조정은 왜 하나요? 이번 예시에서 차이가 아주 크다고 말할 수 있나요?

**Q3.** √d_k로 나누는 것만으로 가중치 합이 1이 되나요? 한 Query의 점수가 모두 같다면 Softmax 가중치는 어떻게 되나요?

#### 제출 결과

가중치 비교 표, 첫 행 최댓값·합, Q1~Q3 답변.


In [5]:
# 크기 조정 없이 Softmax를 적용하세요.
weights_no_scale = softmax_rows(scores)

# 첫 Query 행의 두 가중치 배열을 비교하세요.
compare = pd.DataFrame({"j": [1, 2, 3], "scaled": weights[0], "no_scale": weights_no_scale[0]})
print(compare.round(8).to_string(index=False))
# 각각의 최댓값과 합을 출력하세요.
print("조정 후 최댓값:", round(weights[0].max(), 8), "합:", weights[0].sum())
print("조정 전 최댓값:", round(weights_no_scale[0].max(), 8), "합:", weights_no_scale[0].sum())

 j   scaled  no_scale
 1 0.327556  0.325177
 2 0.334991  0.335665
 3 0.337453  0.339158
조정 후 최댓값: 0.33745273 합: 1.0
조정 전 최댓값: 0.33915807 합: 0.9999999999999999


#### 답변 작성란

**수식 및 계산 과정**

- 작성: `scores`는 그대로 두고 √2로 나누는 단계만 뺀다.
  - 조정 후 첫 행: [0.32755578, 0.33499149, 0.33745273], 최댓값 0.33745273, 합 1
  - 조정 전 첫 행: [0.325177, 0.335665, 0.339158], 최댓값 0.33915807, 합 1

**Q1. 이번에는 왜 √3이나 √1이 아니라 √2로 나누나요? 첫 행 가중치의 최댓값은 조정 전후에 각각 얼마인가요?**

- 답변: 이번 Key 한 행의 성분 수 d_k가 2이므로 √2로 나눈다. 관측 수 3이나 V의 성분 수 1은 기준이 아니다. 첫 행 가중치의 최댓값은 조정 전 0.33915807, 조정 후 0.33745273이다.

**Q2. Scaled Dot-Product Attention에서 크기 조정은 왜 하나요? 이번 예시에서 차이가 아주 크다고 말할 수 있나요?**

- 답변: 차원이 커지면 내적 점수의 크기가 커져 Softmax가 한쪽으로 쏠릴 수 있고 학습에 쓰이는 기울기도 작아질 수 있어, 이를 완화하려고 크기를 조정한다. 이번 예시는 d_k가 2로 작아 최댓값 차이가 약 0.0017뿐이므로 차이가 크다고 말할 수 없다.

**Q3. √d_k로 나누는 것만으로 가중치 합이 1이 되나요? 한 Query의 점수가 모두 같다면 Softmax 가중치는 어떻게 되나요?**

- 답변: 아니다. √d_k로 나누는 것은 점수의 크기만 바꾸며, 합이 1이 되는 것은 Softmax 덕분이다. 한 Query의 점수가 모두 같으면 Softmax 가중치는 모두 같아지며 이번처럼 세 값이면 각각 1/3이다.


---

## 기초 과제. 다음 관측 구간에 Attention 적용하기


### 문제 3-1. 동일한 계산으로 다른 구간 요약하기

#### 문제 설명

`X_home`은 4번째부터 6번째 관측의 기온·상대습도를 같은 배율로 바꾼 입력입니다. WQ·WK·WV와 Softmax 함수를 그대로 사용하여 Self-Attention을 계산하세요.

이번 과제는 **새로운 3행 구간 안에서만** 계산합니다. 실습의 Q·K·V나 출력값을 이어 붙이지 않습니다. 가중치를 학습하거나 조정할 필요는 없습니다.

#### 요구사항

1. `Q_home`, `K_home`, `V_home`을 각각 만드세요.
2. `scores_home`, `scaled_home`, `weights_home`, `output_home`을 순서대로 계산하세요.
3. Q·K·V와 가중치·출력의 값과 크기를 출력하세요. 가중치의 행별 합도 확인하세요.
4. 첫 Query의 첫 내적 점수와 첫 가중합 출력을 수식에 대입해 확인하세요.
5. 첫 Query에서 가장 큰 비중의 관측 시각을 확인하고 출력의 의미를 설명하세요.

#### 작성 도움말

`d_home=K_home.shape[1]`을 사용합니다. `X_home`은 준비 셀에서 이미 배율을 조정했으므로 다시 나누지 않습니다. 첫 관측 시각은 00:40입니다.

#### 해석 질문

**Q1.** Q_home·K_home·V_home·weights_home·output_home의 크기는 각각 무엇인가요? 실습과 같은 크기가 되는 이유는 무엇인가요?

**Q2.** 첫 내적 점수와 첫 가중합 출력은 각각 얼마인가요? 대입 식을 쓰세요.

**Q3.** 첫 Query의 세 가중치와 가장 큰 비중의 시각은 무엇인가요? 전체 가중치의 각 행 합은 얼마인가요?

**Q4.** 결과 숫자가 실습과 달라진 것은 가중치를 새로 학습했기 때문인가요? 이 출력을 미래 기온 예측이라고 설명해도 되나요?

#### 제출 결과

단계별 계산·크기, 행별 가중치 합, 첫 행 대입 계산, Q1~Q4 답변. 모델 학습이나 심화 비교는 요구하지 않습니다.


In [6]:
# X_home에 같은 가중치 행렬을 적용하세요.
Q_home = X_home @ WQ
K_home = X_home @ WK
V_home = X_home @ WV

# 내적, 크기 조정, Softmax, 가중합을 순서대로 계산하세요.
d_home = K_home.shape[1]
scores_home = Q_home @ K_home.T
scaled_home = scores_home / np.sqrt(d_home)
weights_home = softmax_rows(scaled_home)
output_home = weights_home @ V_home

# 각 값과 크기, 행별 합, 첫 Query의 결과를 출력하세요.
print("Q_home:", np.round(Q_home, 6), Q_home.shape)
print("K_home:", np.round(K_home, 6), K_home.shape)
print("V_home:", np.round(V_home, 6), V_home.shape)
print("scores_home:", np.round(scores_home, 6), scores_home.shape)
print("scaled_home:", np.round(scaled_home, 6), scaled_home.shape)
print("weights_home:", np.round(weights_home, 8), weights_home.shape)
print("행별 가중치 합:", weights_home.sum(axis=1))
print("output_home:", np.round(output_home, 6), output_home.shape)
print("첫 Query의 가중치 최대 시각:", df["Date Time"].iloc[3:6].iloc[int(np.argmax(weights_home[0]))])

Q_home: [[-0.831  0.942]
 [-0.827  0.941]
 [-0.805  0.944]] (3, 2)
K_home: [[-0.831   0.471 ]
 [-0.827   0.4705]
 [-0.805   0.472 ]] (3, 2)
V_home: [[-0.831]
 [-0.827]
 [-0.805]] (3, 1)
scores_home: [[1.134243 1.130448 1.113579]
 [1.130448 1.126669 1.109887]
 [1.113579 1.109887 1.093593]] (3, 3)
scaled_home: [[0.802031 0.799347 0.787419]
 [0.799347 0.796676 0.784809]
 [0.787419 0.784809 0.773287]] (3, 3)
weights_home: [[0.33525381 0.33435537 0.33039081]
 [0.33524444 0.33434993 0.33040564]
 [0.33519254 0.33431862 0.33048884]] (3, 3)
행별 가중치 합: [1. 1. 1.]
output_home: [[-0.821072]
 [-0.821072]
 [-0.82107 ]] (3, 1)
첫 Query의 가중치 최대 시각: 01.01.2009 00:40:00


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 첫 행 입력 X_home[0] = [-0.831, 0.942]
  - q1 = [-0.831, 0.942], k1 = [-0.831, 0.471], v1 = [-0.831]
  - 첫 내적 = (-0.831)(-0.831) + (0.942)(0.471) = 0.690561 + 0.443682 = 1.134243
  - A 첫 행 = [0.33525381, 0.33435537, 0.33039081]
  - 첫 출력 = 0.33525381×(-0.831) + 0.33435537×(-0.827) + 0.33039081×(-0.805) ≈ -0.821072

**Q1. Q_home·K_home·V_home·weights_home·output_home의 크기는 각각 무엇인가요? 실습과 같은 크기가 되는 이유는 무엇인가요?**

- 답변: `Q_home`·`K_home`은 (3, 2), `V_home`은 (3, 1), `weights_home`은 (3, 3), `output_home`은 (3, 1)이다. 입력이 똑같이 3행·2열이고 같은 가중치 행렬을 사용하므로 실습과 크기가 같다.

**Q2. 첫 내적 점수와 첫 가중합 출력은 각각 얼마인가요? 대입 식을 쓰세요.**

- 답변: 첫 내적은 (-0.831)×(-0.831) + 0.942×0.471 = 1.134243이다. 첫 출력은 0.33525381×(-0.831) + 0.33435537×(-0.827) + 0.33039081×(-0.805) ≈ -0.821072다.

**Q3. 첫 Query의 세 가중치와 가장 큰 비중의 시각은 무엇인가요? 전체 가중치의 각 행 합은 얼마인가요?**

- 답변: 첫 Query의 가중치는 0.33525381, 0.33435537, 0.33039081이고 가장 큰 비중의 시각은 00:40이다. 전체 가중치의 각 행 합은 1이다.

**Q4. 결과 숫자가 실습과 달라진 것은 가중치를 새로 학습했기 때문인가요? 이 출력을 미래 기온 예측이라고 설명해도 되나요?**

- 답변: 아니다. 가중치 행렬은 그대로이고 입력이 4~6번째 관측으로 바뀌어서 숫자가 달라졌다. 이 출력은 이미 관측한 구간의 Value를 비중으로 합친 값이므로 미래 기온 예측이라고 설명하면 안 된다.


## 제출 전 확인

- 실습 2개와 기초 과제의 요구사항을 모두 수행했나요?
- 각 문제의 모든 해석 질문에 답했나요?
- Q·K·V의 역할과 Attention 가중치를 구분했나요?
- 내적·크기 조정·Softmax·가중합 순서를 지켰나요?
- 각 Query 행의 가중치 합이 1인지 확인했나요?
- 출력을 미래 기온 예측이나 인과관계로 해석하지 않았나요?


---

## 실습 마무리

실습 1·실습 2·기초 과제에서 배운 개념과 계산을 자신의 말로 정리하세요. 새로운 심화 문제가 아니라 **앞에서 사용한 수식과 결과를 다시 확인하는 복습**입니다. exp·제곱근의 값은 제공 코드나 앞의 계산 표를 참고해도 됩니다.

1. Attention과 Self-Attention은 각각 무엇인가요? 이번 계산이 Self-Attention인 이유와, Self가 자기 행만 참고한다는 뜻인지 설명하세요.

2. Q·K·V의 역할과 첫 행의 값, 전체 행렬 크기를 쓰세요. WQ·WK·WV와 Attention 가중치 A의 차이도 설명하세요.

3. 첫 Query와 첫 Key의 내적을 곱셈부터 계산하세요. Q@K.T에서 전치가 필요한 이유와 √2로 나누는 이유를 각각 설명하세요.

4. Softmax는 점수를 어떤 순서로 비중으로 바꾸나요? axis=1과 keepdims=True의 역할, 행별 합, 한 행의 점수가 모두 같을 때의 결과를 설명하세요.

5. 첫 Query의 비중 [0.32755578, 0.33499149, 0.33745273]을 Value [−0.802, −0.841, −0.851]에 적용해 출력을 구하세요. 양수 비중에서 음수 출력이 나오는 이유와 출력 세 개의 의미도 설명하세요.

6. 크기 조정 전후 첫 행의 최대 비중은 어떻게 달라지나요? 과제 구간에서 가장 큰 비중의 시각과 첫 출력을 쓰고, 이 결과가 학습·미래 예측·인과관계를 뜻하는지 설명하세요.


### 실습 마무리 답변 작성란

**1. Attention과 Self-Attention은 각각 무엇인가요? 이번 계산이 Self-Attention인 이유와, Self가 자기 행만 참고한다는 뜻인지 설명하세요.**

- 답변: Attention은 Query와 Key의 비교로 비중을 정해 Value를 합치는 방식이고, Self-Attention은 Q·K·V를 같은 입력 묶음에서 만드는 방식이다. 이번에는 Q·K·V를 모두 같은 X에서 만들었기 때문에 Self-Attention이다. Self는 같은 입력에서 가져온다는 뜻이지 자기 행만 참고한다는 뜻이 아니다.

- 계산 과정 또는 예시: 첫 Query의 비중은 자기 행 0.32755578, 다른 두 행 0.33499149, 0.33745273으로 세 행을 모두 참고한다.

**2. Q·K·V의 역할과 첫 행의 값, 전체 행렬 크기를 쓰세요. WQ·WK·WV와 Attention 가중치 A의 차이도 설명하세요.**

- 답변: Q는 기준, K는 비교 대상, V는 합쳐질 정보다. 첫 행은 q1 = [-0.802, 0.933], k1 = [-0.802, 0.4665], v1 = [-0.802]이고 크기는 Q·K (3, 2), V (3, 1)이다. WQ·WK·WV는 Q·K·V를 만드는 가중치이고, A는 그 결과의 점수로 계산한 반영 비중이다.

- 계산 과정 또는 예시: `Q = X @ WQ`, `K = X @ WK`, `V = X @ WV`, `A = softmax_rows(Q @ K.T / √2)`. W는 입력이 바뀌어도 그대로이고 A는 입력에 따라 달라진다.

**3. 첫 Query와 첫 Key의 내적을 곱셈부터 계산하세요. Q@K.T에서 전치가 필요한 이유와 √2로 나누는 이유를 각각 설명하세요.**

- 답변: 첫 내적은 1.078448이다. Q와 K가 모두 (3, 2)여서 전치로 K를 (2, 3)으로 바꾸어야 모든 Q·K 쌍을 비교하는 (3, 3)을 만들 수 있다. √2로 나누는 이유는 d_k = 2에 맞추어 점수 크기를 조정해 Softmax가 쏠리는 것을 완화하기 위해서다.

- 계산 과정 또는 예시: (-0.802)×(-0.802) + 0.933×0.4665 = 0.643204 + 0.435245 = 1.078448, 조정 후 1.078448/√2 = 0.762578

**4. Softmax는 점수를 어떤 순서로 비중으로 바꾸나요? axis=1과 keepdims=True의 역할, 행별 합, 한 행의 점수가 모두 같을 때의 결과를 설명하세요.**

- 답변: 각 점수에 지수 함수를 적용해 양수로 바꾼 뒤 그 행의 지수값 합으로 나눈다. `axis=1`은 행별로 계산하고 `keepdims=True`는 결과를 세로 한 열 모양으로 유지해 각 행에 맞춰 나누게 한다. 행별 합은 1이며 한 행의 점수가 모두 같으면 비중도 모두 같아져 값이 세 개면 각각 1/3이다.

- 계산 과정 또는 예시: `weights.sum(axis=1)` = [1, 1, 1]. 점수가 [c, c, c]이면 exp(c)/(3×exp(c)) = 1/3이다.

**5. 첫 Query의 비중 [0.32755578, 0.33499149, 0.33745273]을 Value [−0.802, −0.841, −0.851]에 적용해 출력을 구하세요. 양수 비중에서 음수 출력이 나오는 이유와 출력 세 개의 의미도 설명하세요.**

- 답변: 출력은 약 -0.8316이다. 비중은 양수지만 Value가 모두 음수이므로 가중평균인 출력도 음수가 된다. 출력 세 개는 각 Query 행을 기준으로 Value를 합친 요약값이며 미래 기온 예측값이 아니다.

- 계산 과정 또는 예시: 0.32755578×(-0.802) + 0.33499149×(-0.841) + 0.33745273×(-0.851) = -0.262700 - 0.281728 - 0.287172 = -0.831600, `output` = [-0.8316, -0.831612, -0.831615]

**6. 크기 조정 전후 첫 행의 최대 비중은 어떻게 달라지나요? 과제 구간에서 가장 큰 비중의 시각과 첫 출력을 쓰고, 이 결과가 학습·미래 예측·인과관계를 뜻하는지 설명하세요.**

- 답변: 첫 행의 최대 비중은 조정 전 0.33915807, 조정 후 0.33745273이다. 과제 구간에서 가장 큰 비중의 시각은 00:40이고 첫 출력은 약 -0.821072다. 이 결과는 고정 가중치로 한 번 계산한 것이므로 학습, 미래 예측, 인과관계 중 어느 것도 뜻하지 않는다.

- 계산 과정 또는 예시: 과제 첫 행 비중은 [0.33525381, 0.33435537, 0.33039081]이다. 가중치를 업데이트하는 코드도, 미래 정답과 비교하는 코드도 없었다.
